# LYS pre-Gd T1 brain masking — animal-grouped 16/2/2 holdout

This notebook trains one standard 3-D nnU-Net on 16 corrected scans, selects
`checkpoint_best.pth` using 2 validation scans, and evaluates once on 2 sealed
test scans. Splits are grouped by animal: C24S5 is validation and C25S1 is
test. The other 16 scans are training data.

Attach `LYS_T1_brainmask_manual_holdout_16_2_2_20260731.zip` (or its unpacked
Kaggle dataset), enable **GPU T4 ×2** and Internet, and run all cells. Test
images and labels are excluded from nnU-Net planning, preprocessing, training,
and checkpoint selection.

## 1 — Install pinned nnU-Net before loading the scientific stack

Installation runs before NumPy, SciPy, or scikit-image are imported. This
prevents a live Kaggle kernel from mixing binary extensions from the base
image with packages selected by pip.

In [ ]:
import importlib.metadata
import inspect
import shutil
import subprocess
import sys
from pathlib import Path

NNUNET_COMMIT = "468cf803df9b267150ae2b6c0c59b8ac84f16227"
NUM_GPUS = 2
BASE_NUMPY_VERSION = importlib.metadata.version("numpy")
BASE_SCIPY_VERSION = importlib.metadata.version("scipy")
BASE_SKIMAGE_VERSION = importlib.metadata.version("scikit-image")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"numpy=={BASE_NUMPY_VERSION}",
        f"scipy=={BASE_SCIPY_VERSION}",
        f"scikit-image=={BASE_SKIMAGE_VERSION}",
        f"git+https://github.com/MIC-DKFZ/nnUNet.git@{NNUNET_COMMIT}",
        "nibabel",
        "pandas",
        "scipy",
        "surface-distance",
        "matplotlib",
    ],
    check=True,
)
import torch
from IPython.display import display

assert importlib.metadata.version("nnunetv2") == "2.8.1"
assert torch.cuda.is_available(), "Enable a Kaggle GPU"
assert torch.cuda.device_count() >= NUM_GPUS, (
    torch.cuda.device_count(), NUM_GPUS
)
supported_arches = set(torch.cuda.get_arch_list())
gpu_records = []
for index in range(NUM_GPUS):
    major, minor = torch.cuda.get_device_capability(index)
    architecture = f"sm_{major}{minor}"
    assert architecture in supported_arches, (
        f"{torch.cuda.get_device_name(index)} uses {architecture}, "
        f"but this PyTorch build supports {sorted(supported_arches)}"
    )
    gpu_records.append({
        "index": index,
        "name": torch.cuda.get_device_name(index),
        "capability": f"{major}.{minor}",
    })

subprocess.run(["nvidia-smi"], check=True)
for command in (
    "nnUNetv2_plan_and_preprocess",
    "nnUNetv2_train",
    "nnUNetv2_predict_from_modelfolder",
):
    assert shutil.which(command), command
    subprocess.run(
        [command, "--help"],
        check=True,
        stdout=subprocess.DEVNULL,
    )

import nnunetv2
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

print("nnU-Net package:", Path(nnunetv2.__file__).resolve())
print("Base trainer:", inspect.getfile(nnUNetTrainer))
display({
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "nnunet": importlib.metadata.version("nnunetv2"),
    "nnunet_commit": NNUNET_COMMIT,
    "gpus": gpu_records,
})

## 2 — Configuration

The execution flags run exactly one predefined development split. Keep the
resume archive enabled so a later Kaggle session can continue from the latest
completed epoch.

In [ ]:
import hashlib
import importlib.metadata
import inspect
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, Image, display

RUN_SEED = 20260731
NNUNET_COMMIT = "468cf803df9b267150ae2b6c0c59b8ac84f16227"  # v2.8.1
DATASET_ID = 503
DATASET_NAME = "Dataset503_LYST1BrainMaskHoldoutV1"
PLANNER = "ExperimentPlanner"
PLANS = "nnUNetPlans"
CONFIGURATION = "3d_fullres"
BENCHMARK_TRAINER = "nnUNetTrainer_5epochs_SaveEveryEpoch"
CV_TRAINER = "nnUNetTrainer_250epochs_SaveEveryEpoch"

RUN_BENCHMARK_5E = False
RUN_HOLDOUT_250 = True
RUN_FINAL_ALL_250 = False
FOLDS_TO_RUN = [0]
NUM_GPUS = 2
BUILD_RESUME_ARCHIVE = True

# Set only if automatic discovery finds the wrong item.
TRAINING_PACKAGE_OVERRIDE = None  # package root, manifest, or .zip
RESUME_ARCHIVE_OVERRIDE = None    # ...standard3d_resume.tar.gz

# Change these only after an explicit, documented inclusion correction.
EXPECTED_APPROVED_CASES = 20
EXPECTED_ANIMAL_GROUPS = 10
EXPECTED_SPLIT_COUNTS = {"train": 16, "validation": 2, "test": 2}
SURFACE_TOLERANCE_MM = 0.15
DEPLOY_DISABLE_TTA = True

assert set(FOLDS_TO_RUN) <= set(range(5))
assert len(FOLDS_TO_RUN) == len(set(FOLDS_TO_RUN))
assert NUM_GPUS in (1, 2)

WORK = Path("/kaggle/working")
EXPERIMENT_ROOT = WORK / "LYS_T1_brainmask_standard3d_holdout_16_2_2"
PROVENANCE = EXPERIMENT_ROOT / "provenance"
RUNS = EXPERIMENT_ROOT / "runs"
NNUNET_RAW = WORK / "nnUNet_raw"
NNUNET_PREPROCESSED = WORK / "nnUNet_preprocessed"
NNUNET_RESULTS = WORK / "nnUNet_results"
for key, value in {
    "nnUNet_raw": NNUNET_RAW,
    "nnUNet_preprocessed": NNUNET_PREPROCESSED,
    "nnUNet_results": NNUNET_RESULTS,
}.items():
    os.environ[key] = str(value)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

display({
    "benchmark_5e": RUN_BENCHMARK_5E,
    "holdout_250": RUN_HOLDOUT_250,
    "final_all_250": RUN_FINAL_ALL_250,
    "folds": FOLDS_TO_RUN,
    "num_gpus": NUM_GPUS,
    "deploy_disable_tta": DEPLOY_DISABLE_TTA,
})

## 3 — Restore optional resume state

Attach at most one prior resume archive. It contains checkpoints and
reports, never source T1 images or manual masks.

In [ ]:
def safe_extract_tar(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with tarfile.open(archive_path, "r:*") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            assert target == root or root in target.parents, member.name
        archive.extractall(destination)

if RESUME_ARCHIVE_OVERRIDE:
    resume_archives = [Path(RESUME_ARCHIVE_OVERRIDE)]
else:
    resume_archives = sorted(
        Path("/kaggle/input").rglob(
            "LYS_T1_brainmask_holdout_16_2_2_resume.tar.gz"
        )
    )
assert len(resume_archives) <= 1, resume_archives
if resume_archives:
    assert not EXPERIMENT_ROOT.exists()
    assert not (NNUNET_RESULTS / DATASET_NAME).exists()
    safe_extract_tar(resume_archives[0], WORK)
    print("Restored:", resume_archives[0])
else:
    print("No resume archive attached.")

PROVENANCE.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)

In [ ]:
# Kaggle may expand nested .nii.gz files to .nii while leaving the manifest
# paths unchanged. Recreate a private working copy with valid .nii.gz files.
uploaded_manifests = sorted(
    Path("/kaggle/input").rglob("training_manifest.csv")
)
if len(uploaded_manifests) == 1:
    uploaded_manifest = uploaded_manifests[0]
    uploaded_root = uploaded_manifest.parent
    uploaded_rows = pd.read_csv(uploaded_manifest, keep_default_na=False)
    needs_normalization = any(
        not (uploaded_root / str(value)).is_file()
        and str(value).endswith(".nii.gz")
        and (uploaded_root / str(value)).with_suffix("").is_file()
        for column in ("image", "mask")
        for value in uploaded_rows[column]
    )
    if needs_normalization:
        normalized_root = WORK / "normalized_training_package"
        normalized_root.mkdir(parents=True, exist_ok=True)
        normalized_rows = uploaded_rows.copy()
        for column, hash_column in (
            ("image", "image_sha256"),
            ("mask", "mask_sha256"),
        ):
            original_hash_column = f"source_manifest_{hash_column}"
            normalized_rows[original_hash_column] = normalized_rows[
                hash_column
            ]
            for index, value in normalized_rows[column].items():
                relative_path = Path(str(value))
                source = uploaded_root / relative_path
                if not source.is_file() and str(source).endswith(".nii.gz"):
                    source = source.with_suffix("")
                assert source.is_file(), source
                destination = normalized_root / relative_path
                destination.parent.mkdir(parents=True, exist_ok=True)
                nib.save(nib.load(str(source)), str(destination))
                normalized_rows.at[index, hash_column] = sha256(destination)
        normalized_manifest = normalized_root / "training_manifest.csv"
        normalized_rows.to_csv(normalized_manifest, index=False)
        TRAINING_PACKAGE_OVERRIDE = normalized_manifest
        print("Normalized Kaggle .nii files:", normalized_manifest)
    else:
        print("NIfTI upload paths already match the manifest.")
else:
    print("NIfTI normalization deferred to package discovery.")

## 4 — Locate and validate the split-aware package

The manifest fixes the animal-grouped split. Test rows must be approved but
must have `include_for_nnunet=no`; the notebook still verifies their files and
hashes before keeping them in a separate sealed-test folder.

In [ ]:
REQUIRED_COLUMNS = {
    "case_id",
    "animal_id",
    "split",
    "modality",
    "acquisition_role",
    "image",
    "mask",
    "include_for_nnunet",
    "mask_review",
    "reviewer",
    "reviewed_at",
    "image_sha256",
    "mask_sha256",
}
TRUE_VALUES = {"1", "true", "yes", "y"}
PASS_VALUES = {"pass", "passed", "approved", "approve", "accepted"}

def safe_extract_zip(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)

def locate_manifest() -> Path:
    if TRAINING_PACKAGE_OVERRIDE:
        candidate = Path(TRAINING_PACKAGE_OVERRIDE)
        if candidate.is_file() and candidate.name == "training_manifest.csv":
            return candidate
        if candidate.is_dir():
            matches = sorted(candidate.rglob("training_manifest.csv"))
            assert len(matches) == 1, matches
            return matches[0]
        assert candidate.suffix.lower() == ".zip", candidate
        extraction = WORK / "uploaded_training_package"
        extraction.mkdir(parents=True, exist_ok=True)
        safe_extract_zip(candidate, extraction)
        matches = sorted(extraction.rglob("training_manifest.csv"))
        assert len(matches) == 1, matches
        return matches[0]

    matches = sorted(Path("/kaggle/input").rglob("training_manifest.csv"))
    if len(matches) == 1:
        return matches[0]
    assert not matches, f"Multiple training manifests: {matches}"
    archives = sorted(
        Path("/kaggle/input").rglob(
            "LYS_T1_brainmask_manual_holdout_16_2_2_20260731.zip"
        )
    )
    assert len(archives) == 1, (
        "Attach exactly one approved T1 brain-mask package; "
        f"found {archives}"
    )
    extraction = WORK / "uploaded_training_package"
    extraction.mkdir(parents=True, exist_ok=True)
    safe_extract_zip(archives[0], extraction)
    matches = sorted(extraction.rglob("training_manifest.csv"))
    assert len(matches) == 1, matches
    return matches[0]

def relative_member(root: Path, value: str) -> Path:
    text = str(value).strip()
    assert text and not Path(text).is_absolute(), text
    candidate = (root / text).resolve()
    resolved_root = root.resolve()
    assert candidate == resolved_root or resolved_root in candidate.parents
    assert candidate.is_file(), candidate
    return candidate

TRAINING_MANIFEST = locate_manifest()
PACKAGE_ROOT = TRAINING_MANIFEST.parent
all_rows = pd.read_csv(TRAINING_MANIFEST, keep_default_na=False)
assert REQUIRED_COLUMNS <= set(all_rows.columns), (
    REQUIRED_COLUMNS - set(all_rows.columns)
)
assert all_rows.case_id.is_unique

include = (
    all_rows.include_for_nnunet.astype(str).str.strip().str.lower()
    .isin(TRUE_VALUES)
)
approved = (
    all_rows.mask_review.astype(str).str.strip().str.lower()
    .isin(PASS_VALUES)
)
rows = all_rows.loc[approved].copy().reset_index(drop=True)
rows["split"] = rows["split"].astype(str).str.strip().str.lower()
assert len(rows) == EXPECTED_APPROVED_CASES, (
    len(rows), EXPECTED_APPROVED_CASES
)
assert rows.animal_id.astype(str).str.strip().ne("").all()
assert rows.animal_id.nunique() == EXPECTED_ANIMAL_GROUPS, (
    rows.animal_id.nunique(), EXPECTED_ANIMAL_GROUPS
)
assert rows.reviewer.astype(str).str.strip().ne("").all()
reviewed_times = pd.to_datetime(rows.reviewed_at, utc=True, errors="coerce")
assert reviewed_times.notna().all(), "Missing/invalid reviewed_at"
assert (
    rows.modality.astype(str).str.strip().str.lower() == "t1w"
).all()
assert (
    rows.acquisition_role.astype(str).str.strip().str.lower()
    == "pre_gd"
).all()
assert rows["split"].value_counts().to_dict() == EXPECTED_SPLIT_COUNTS
expected_include = all_rows["split"].astype(str).str.lower() != "test"
assert (include == expected_include).all(), (
    "Only train/validation rows may have include_for_nnunet=yes"
)
animal_split_counts = rows.groupby("animal_id")["split"].nunique()
assert (animal_split_counts == 1).all(), "Animal leakage across splits"
assert set(rows.loc[rows["split"] == "validation", "animal_id"]) == {"C24S5"}
assert set(rows.loc[rows["split"] == "test", "animal_id"]) == {"C25S1"}

geometry_records = []
verified_paths = {}
for record in rows.to_dict("records"):
    case_id = str(record["case_id"]).strip()
    assert case_id
    image_path = relative_member(PACKAGE_ROOT, record["image"])
    mask_path = relative_member(PACKAGE_ROOT, record["mask"])
    assert sha256(image_path) == str(record["image_sha256"]).lower()
    assert sha256(mask_path) == str(record["mask_sha256"]).lower()
    image = nib.load(str(image_path))
    mask = nib.load(str(mask_path))
    assert image.ndim == mask.ndim == 3, case_id
    assert image.shape == mask.shape, case_id
    assert np.allclose(image.affine, mask.affine, atol=1e-5), case_id
    image_data = np.asarray(image.dataobj)
    mask_data = np.asarray(mask.dataobj)
    assert np.isfinite(image_data).all(), case_id
    values = np.unique(mask_data)
    assert set(values.tolist()) <= {0, 1}, (case_id, values)
    assert np.count_nonzero(mask_data) > 0, case_id
    verified_paths[case_id] = (image_path, mask_path)
    geometry_records.append({
        "case_id": case_id,
        "animal_id": record["animal_id"],
        "split": record["split"],
        "shape": "x".join(map(str, image.shape)),
        "spacing_mm": "x".join(
            f"{value:.9g}" for value in image.header.get_zooms()[:3]
        ),
        "brain_voxels": int(np.count_nonzero(mask_data)),
        "image_sha256": sha256(image_path),
        "mask_sha256": sha256(mask_path),
    })
    del image_data, mask_data

input_manifest_copy = PROVENANCE / "training_manifest.csv"
if input_manifest_copy.is_file():
    assert sha256(input_manifest_copy) == sha256(TRAINING_MANIFEST)
else:
    shutil.copy2(TRAINING_MANIFEST, input_manifest_copy)
geometry = pd.DataFrame(geometry_records)
geometry.to_csv(PROVENANCE / "input_geometry.csv", index=False)
display({
    "manifest": str(TRAINING_MANIFEST),
    "approved_cases": len(rows),
    "split_counts": rows["split"].value_counts().to_dict(),
    "animal_groups": rows.animal_id.nunique(),
    "shapes": geometry["shape"].value_counts().to_dict(),
    "spacings_mm": geometry["spacing_mm"].value_counts().to_dict(),
})

## 5 — Materialize only train/validation data and one fixed split

The 18 development scans become nnU-Net `imagesTr`/`labelsTr`. The 2 test
scans remain outside the raw and preprocessed nnU-Net datasets.

In [ ]:
RAW_DATASET = NNUNET_RAW / DATASET_NAME
images_tr = RAW_DATASET / "imagesTr"
labels_tr = RAW_DATASET / "labelsTr"
images_tr.mkdir(parents=True, exist_ok=True)
labels_tr.mkdir(parents=True, exist_ok=True)
SEALED_TEST = WORK / "LYS_T1_sealed_test"
test_images = SEALED_TEST / "images"
test_labels = SEALED_TEST / "labels"
test_images.mkdir(parents=True, exist_ok=True)
test_labels.mkdir(parents=True, exist_ok=True)

mapping_records = []
development = rows.loc[rows["split"] != "test"].sort_values("case_id")
for index, record in enumerate(development.to_dict("records")):
    case_id = str(record["case_id"])
    nnunet_case_id = f"T1BM_{index:03d}"
    source_image, source_mask = verified_paths[case_id]
    output_image = images_tr / f"{nnunet_case_id}_0000.nii.gz"
    output_mask = labels_tr / f"{nnunet_case_id}.nii.gz"
    if not output_image.is_file():
        shutil.copy2(source_image, output_image)
    mask_image = nib.load(str(source_mask))
    binary = (np.asarray(mask_image.dataobj) > 0).astype(np.uint8)
    if not output_mask.is_file():
        header = mask_image.header.copy()
        header.set_data_dtype(np.uint8)
        clean_mask = nib.Nifti1Image(binary, mask_image.affine, header=header)
        qform, qcode = mask_image.get_qform(coded=True)
        sform, scode = mask_image.get_sform(coded=True)
        clean_mask.set_qform(qform, int(qcode))
        clean_mask.set_sform(sform, int(scode))
        nib.save(clean_mask, str(output_mask))
    mapping_records.append({
        "case_id": case_id,
        "nnunet_case_id": nnunet_case_id,
        "animal_id": str(record["animal_id"]),
        "split": str(record["split"]),
        "reviewer": str(record["reviewer"]),
        "reviewed_at": str(record["reviewed_at"]),
        "source_image_sha256": str(record["image_sha256"]).lower(),
        "source_mask_sha256": str(record["mask_sha256"]).lower(),
    })

test_mapping_records = []
test_rows = rows.loc[rows["split"] == "test"].sort_values("case_id")
for index, record in enumerate(test_rows.to_dict("records")):
    case_id = str(record["case_id"])
    test_id = f"T1TEST_{index:03d}"
    source_image, source_mask = verified_paths[case_id]
    destination_image = test_images / f"{test_id}_0000.nii.gz"
    destination_mask = test_labels / f"{test_id}.nii.gz"
    if not destination_image.is_file():
        shutil.copy2(source_image, destination_image)
    if not destination_mask.is_file():
        shutil.copy2(source_mask, destination_mask)
    test_mapping_records.append({
        "case_id": case_id,
        "nnunet_case_id": test_id,
        "animal_id": str(record["animal_id"]),
        "split": "test",
        "source_image_sha256": str(record["image_sha256"]).lower(),
        "source_mask_sha256": str(record["mask_sha256"]).lower(),
    })

dataset_json = {
    "channel_names": {"0": "pre-Gd T1w"},
    "labels": {"background": 0, "brain": 1},
    "numTraining": len(mapping_records),
    "file_ending": ".nii.gz",
}
(RAW_DATASET / "dataset.json").write_text(
    json.dumps(dataset_json, indent=2, sort_keys=True) + "\n"
)
mapping = pd.DataFrame(mapping_records)
test_mapping = pd.DataFrame(test_mapping_records)
mapping.to_csv(RAW_DATASET / "case_mapping.csv", index=False)
test_mapping.to_csv(PROVENANCE / "test_case_mapping.csv", index=False)

training_ids = sorted(
    mapping.loc[mapping["split"] == "train", "nnunet_case_id"]
)
validation_ids = sorted(
    mapping.loc[mapping["split"] == "validation", "nnunet_case_id"]
)
assert len(training_ids) == 16 and len(validation_ids) == 2
assert set(training_ids).isdisjoint(validation_ids)
splits = [{"train": training_ids, "val": validation_ids}]
(RAW_DATASET / "splits_final.json").write_text(
    json.dumps(splits, indent=2, sort_keys=True) + "\n"
)
split_assignments = pd.concat([mapping, test_mapping], ignore_index=True)
assert split_assignments["split"].value_counts().to_dict() == (
    EXPECTED_SPLIT_COUNTS
)
assert (
    split_assignments.groupby("animal_id")["split"].nunique() == 1
).all()
split_assignments.to_csv(
    PROVENANCE / "split_assignments.csv", index=False
)
for name in ("case_mapping.csv", "dataset.json", "splits_final.json"):
    shutil.copy2(RAW_DATASET / name, PROVENANCE / name)
display(
    split_assignments.groupby("split").agg(
        cases=("case_id", "count"),
        animals=("animal_id", "nunique"),
    )
)

## 6 — Plan and preprocess the official compact 3-D candidate

The architecture gate forbids an accidental ResEnc rerun. On the
audited 34-case geometry, the expected plan has a batch size of 2,
patch 80×192×160, 30,785,994 parameters, and about 117 MiB of
float32 inference weights.

In [ ]:
PREPROCESSED_DATASET = NNUNET_PREPROCESSED / DATASET_NAME
PLAN_PATH = PREPROCESSED_DATASET / f"{PLANS}.json"
if not PLAN_PATH.is_file():
    subprocess.run(
        [
            "nnUNetv2_plan_and_preprocess",
            "-d",
            str(DATASET_ID),
            "-pl",
            PLANNER,
            "-c",
            CONFIGURATION,
            "-npfp",
            "2",
            "-np",
            "2",
            "--verify_dataset_integrity",
        ],
        check=True,
    )
assert PLAN_PATH.is_file()
shutil.copy2(
    RAW_DATASET / "splits_final.json",
    PREPROCESSED_DATASET / "splits_final.json",
)

from nnunetv2.utilities.get_network_from_plans import (
    get_network_from_plans,
)

plans_json = json.loads(PLAN_PATH.read_text())
planned = plans_json["configurations"][CONFIGURATION]
architecture = planned["architecture"]
assert architecture["network_class_name"].endswith("PlainConvUNet")
assert "ResidualEncoderUNet" not in architecture["network_class_name"]
assert planned["batch_size"] >= NUM_GPUS, (
    "The global batch must be at least the DDP world size"
)
network = get_network_from_plans(
    architecture["network_class_name"],
    architecture["arch_kwargs"],
    architecture["_kw_requires_import"],
    input_channels=1,
    output_channels=2,
    allow_init=False,
    deep_supervision=False,
)
parameter_count = sum(value.numel() for value in network.parameters())
del network
assert parameter_count <= 35_000_000, parameter_count
plan_summary = {
    "network_class": architecture["network_class_name"],
    "parameter_count": parameter_count,
    "float32_weight_mib": parameter_count * 4 / 1024**2,
    "batch_size": planned["batch_size"],
    "patch_size": planned["patch_size"],
    "spacing": planned["spacing"],
    "median_image_size_in_voxels": (
        planned["median_image_size_in_voxels"]
    ),
    "features_per_stage": (
        architecture["arch_kwargs"]["features_per_stage"]
    ),
    "plans_sha256": sha256(PLAN_PATH),
}
(PROVENANCE / "plan_summary.json").write_text(
    json.dumps(plan_summary, indent=2, sort_keys=True) + "\n"
)

identity = {
    "protocol": "lys_t1_brainmask_standard3d_holdout_16_2_2_v1",
    "run_seed": RUN_SEED,
    "approved_case_count": len(rows),
    "development_case_count": len(mapping),
    "sealed_test_case_count": len(test_mapping),
    "animal_group_count": int(rows.animal_id.nunique()),
    "training_manifest_sha256": sha256(TRAINING_MANIFEST),
    "split_sha256": sha256(PROVENANCE / "splits_final.json"),
    "nnunet_version": importlib.metadata.version("nnunetv2"),
    "nnunet_source_commit": NNUNET_COMMIT,
    "torch_version": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpus": gpu_records,
    "planner": PLANNER,
    "plans": PLANS,
    "configuration": CONFIGURATION,
    "postprocessing": "none",
    "released_model_count": 1,
    "human_review_required": True,
}
identity_path = PROVENANCE / "protocol_identity.json"
if identity_path.is_file():
    assert json.loads(identity_path.read_text()) == identity
else:
    identity_path.write_text(
        json.dumps(identity, indent=2, sort_keys=True) + "\n"
    )
display(plan_summary)

## 7 — Install resume-safe trainer variants

Upstream nnU-Net saves `checkpoint_latest.pth` every 50 epochs. These
otherwise unchanged 5- and 250-epoch variants overwrite that latest
checkpoint after every epoch. This improves crash recovery without
changing the network or loss.

In [ ]:
from nnunetv2.training.nnUNetTrainer import nnUNetTrainer as trainer_package

trainer_directory = Path(trainer_package.__file__).resolve().parent
custom_trainer_file = trainer_directory / "lys_t1_save_every_epoch.py"
custom_trainer_source = """import importlib


length_trainers = importlib.import_module(
    "nnunetv2.training.nnUNetTrainer.variants.training_length."
    "nnUNetTrainer_Xepochs"
)


class nnUNetTrainer_5epochs_SaveEveryEpoch(
    length_trainers.nnUNetTrainer_5epochs
):
    def __init__(self, plans, configuration, fold, dataset_json, device):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.save_every = 1


class nnUNetTrainer_250epochs_SaveEveryEpoch(
    length_trainers.nnUNetTrainer_250epochs
):
    def __init__(self, plans, configuration, fold, dataset_json, device):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.save_every = 1
"""
custom_trainer_file.write_text(custom_trainer_source)
(PROVENANCE / "custom_trainer.py").write_text(custom_trainer_source)
print("Custom trainer:", custom_trainer_file)
print("SHA-256:", sha256(custom_trainer_file))

## 8 — Resume-safe training helpers

A stopped fold continues from its per-epoch latest checkpoint. A
completed development fold is accepted only when its best checkpoint,
summary, and exact expected validation IDs are present.

In [ ]:
def model_root(trainer: str) -> Path:
    return (
        NNUNET_RESULTS
        / DATASET_NAME
        / f"{trainer}__{PLANS}__{CONFIGURATION}"
    )

def model_folder(trainer: str, fold) -> Path:
    return model_root(trainer) / f"fold_{fold}"

def expected_validation_ids(fold: int) -> set[str]:
    split = json.loads(
        (PROVENANCE / "splits_final.json").read_text()
    )[fold]
    return set(split["val"])

def training_command(trainer: str, fold) -> list[str]:
    return [
        "nnUNetv2_train",
        str(DATASET_ID),
        CONFIGURATION,
        str(fold),
        "-tr",
        trainer,
        "-p",
        PLANS,
        "-num_gpus",
        str(NUM_GPUS),
    ]

def train_and_validate(trainer: str, fold: int, marker: Path) -> dict:
    folder = model_folder(trainer, fold)
    best = folder / "checkpoint_best.pth"
    final = folder / "checkpoint_final.pth"
    latest = folder / "checkpoint_latest.pth"
    validation = folder / "validation"
    expected = expected_validation_ids(fold)
    if marker.is_file():
        recorded = json.loads(marker.read_text())
        assert best.is_file()
        assert recorded["checkpoint_best_sha256"] == sha256(best)
        assert {path.stem for path in validation.glob("*.npz")} == expected
        return recorded

    command = training_command(trainer, fold)
    started = time.time()
    if final.is_file():
        subprocess.run(
            command + ["--val", "--npz", "--val_best"],
            check=True,
        )
    else:
        if latest.is_file():
            command.append("--c")
        subprocess.run(
            command + ["--npz", "--val_best"],
            check=True,
        )
    elapsed = time.time() - started

    assert best.is_file()
    assert (validation / "summary.json").is_file()
    observed = {path.stem for path in validation.glob("*.npz")}
    assert observed == expected, {
        "missing": sorted(expected - observed),
        "unexpected": sorted(observed - expected),
    }
    checkpoint = torch.load(
        best, map_location="cpu", weights_only=False
    )
    parameters = sum(
        tensor.numel()
        for tensor in checkpoint["network_weights"].values()
    )
    record = {
        "trainer": trainer,
        "fold": fold,
        "elapsed_seconds_this_call": elapsed,
        "checkpoint_best_sha256": sha256(best),
        "checkpoint_best_bytes": best.stat().st_size,
        "parameter_count": parameters,
        "n_validation_cases": len(expected),
        "checkpoint_selection": "best validation EMA pseudo-Dice",
        "postprocessing": "none",
    }
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text(
        json.dumps(record, indent=2, sort_keys=True) + "\n"
    )
    return record

def train_all(marker: Path) -> dict:
    folder = model_folder(CV_TRAINER, "all")
    final = folder / "checkpoint_final.pth"
    latest = folder / "checkpoint_latest.pth"
    if marker.is_file():
        recorded = json.loads(marker.read_text())
        assert final.is_file()
        assert recorded["checkpoint_final_sha256"] == sha256(final)
        return recorded
    command = training_command(CV_TRAINER, "all")
    started = time.time()
    if not final.is_file():
        if latest.is_file():
            command.append("--c")
        subprocess.run(command, check=True)
    record = {
        "trainer": CV_TRAINER,
        "fold": "all",
        "elapsed_seconds_this_call": time.time() - started,
        "checkpoint_final_sha256": sha256(final),
        "checkpoint_final_bytes": final.stat().st_size,
        "training_cases": len(mapping),
        "release_checkpoint": "checkpoint_final",
        "development_validation_claim": False,
    }
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text(
        json.dumps(record, indent=2, sort_keys=True) + "\n"
    )
    return record

## 9 — Define inference-only checkpoint packaging

This helper removes optimizer, scheduler, scaler, and logger state from the
selected checkpoint without changing network weights.

In [ ]:
def portable_checkpoint(
    source: Path, destination: Path, portable_trainer: str
) -> dict:
    checkpoint = torch.load(
        source, map_location="cpu", weights_only=False
    )
    required = {
        "network_weights",
        "init_args",
        "inference_allowed_mirroring_axes",
    }
    assert required <= set(checkpoint)
    compact = {
        "network_weights": checkpoint["network_weights"],
        "trainer_name": portable_trainer,
        "init_args": checkpoint["init_args"],
        "inference_allowed_mirroring_axes": (
            checkpoint["inference_allowed_mirroring_axes"]
        ),
    }
    destination.parent.mkdir(parents=True, exist_ok=True)
    torch.save(compact, destination)
    return {
        "source_sha256": sha256(source),
        "portable_sha256": sha256(destination),
        "portable_bytes": destination.stat().st_size,
        "preserved_keys": sorted(compact),
    }

## 10 — Train the single 16/2 development split

Only fold 0 exists: 16 training scans and 2 validation scans. The best
checkpoint is selected from validation performance; test data remain sealed.

In [ ]:
holdout_record = None
if RUN_HOLDOUT_250:
    holdout_record = train_and_validate(
        CV_TRAINER,
        0,
        RUNS / "holdout_250/fold_0/complete.json",
    )
    display(holdout_record)
else:
    print("Holdout training disabled.")

## 11 — Validation and sealed-test evaluation

The selected validation checkpoint predicts the test images only after
training is complete. Metrics are reported per case and pooled; the two test
scans belong to one animal and therefore represent one independent test unit.

In [ ]:
EVALUATION_ROOT = EXPERIMENT_ROOT / "holdout_evaluation"

def predict_cases(
    input_folder: Path,
    output_folder: Path,
    expected: set[str],
    disable_tta: bool,
) -> Path:
    observed = {
        path.name.removesuffix(".nii.gz")
        for path in output_folder.glob("*.nii.gz")
    }
    if observed != expected:
        command = [
            "nnUNetv2_predict_from_modelfolder",
            "-i", str(input_folder),
            "-o", str(output_folder),
            "-m", str(model_root(CV_TRAINER)),
            "-f", "0",
            "-chk", "checkpoint_best.pth",
            "-device", "cuda",
            "-npp", "2",
            "-nps", "2",
        ]
        if disable_tta:
            command.append("--disable_tta")
        subprocess.run(command, check=True)
    observed = {
        path.name.removesuffix(".nii.gz")
        for path in output_folder.glob("*.nii.gz")
    }
    assert observed == expected, (observed, expected)
    return output_folder

def case_metrics(target_path: Path, prediction_path: Path) -> dict:
    from surface_distance import metrics as surface_metrics

    target_image = nib.load(str(target_path))
    prediction_image = nib.load(str(prediction_path))
    assert target_image.shape == prediction_image.shape
    assert np.allclose(
        target_image.affine, prediction_image.affine, atol=1e-4
    )
    target = np.asarray(target_image.dataobj) > 0
    prediction = np.asarray(prediction_image.dataobj) > 0
    tp = int(np.count_nonzero(target & prediction))
    fp = int(np.count_nonzero(~target & prediction))
    fn = int(np.count_nonzero(target & ~prediction))
    dice = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0
    precision = tp / (tp + fp) if tp + fp else 1.0
    recall = tp / (tp + fn) if tp + fn else 1.0
    target_voxels = int(np.count_nonzero(target))
    predicted_voxels = int(np.count_nonzero(prediction))
    volume_error_pct = (
        100 * (predicted_voxels - target_voxels) / target_voxels
    )
    spacing = target_image.header.get_zooms()[:3]
    distances = surface_metrics.compute_surface_distances(
        target, prediction, spacing
    )
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "dice": dice,
        "precision": precision,
        "recall": recall,
        "volume_error_pct": volume_error_pct,
        "abs_volume_error_pct": abs(volume_error_pct),
        "hd95_mm": surface_metrics.compute_robust_hausdorff(
            distances, 95
        ),
        "surface_dice": (
            surface_metrics.compute_surface_dice_at_tolerance(
                distances, SURFACE_TOLERANCE_MM
            )
        ),
        "empty_prediction": predicted_voxels == 0,
    }

def summarize_metrics(part: pd.DataFrame) -> dict:
    tp = int(part.tp.sum())
    fp = int(part.fp.sum())
    fn = int(part.fn.sum())
    return {
        "cases": len(part),
        "pooled_voxel_dice": 2 * tp / (2 * tp + fp + fn),
        "mean_case_dice": float(part.dice.mean()),
        "median_case_dice": float(part.dice.median()),
        "minimum_case_dice": float(part.dice.min()),
        "mean_precision": float(part.precision.mean()),
        "mean_recall": float(part.recall.mean()),
        "mean_abs_volume_error_pct": float(
            part.abs_volume_error_pct.mean()
        ),
        "median_hd95_mm": float(part.hd95_mm.median()),
        "mean_surface_dice": float(part.surface_dice.mean()),
        "empty_predictions": int(part.empty_prediction.sum()),
    }


if RUN_HOLDOUT_250 and (
    RUNS / "holdout_250/fold_0/complete.json"
).is_file():
    metric_records = []
    mapping_by_id = mapping.set_index("nnunet_case_id")
    validation_folder = model_folder(CV_TRAINER, 0) / "validation"
    for nnunet_case_id in expected_validation_ids(0):
        identity_row = mapping_by_id.loc[nnunet_case_id]
        metric_records.append({
            "case_id": identity_row.case_id,
            "animal_id": identity_row.animal_id,
            "nnunet_case_id": nnunet_case_id,
            "split": "validation",
            "inference_policy": "default_tta",
            **case_metrics(
                labels_tr / f"{nnunet_case_id}.nii.gz",
                validation_folder / f"{nnunet_case_id}.nii.gz",
            ),
        })

    expected_test = set(test_mapping.nnunet_case_id)
    for policy, disable_tta in (
        ("default_tta", False),
        ("no_tta", True),
    ):
        prediction_folder = predict_cases(
            test_images,
            EVALUATION_ROOT / f"test_{policy}",
            expected_test,
            disable_tta,
        )
        for record in test_mapping.to_dict("records"):
            nnunet_case_id = record["nnunet_case_id"]
            metric_records.append({
                **record,
                "inference_policy": policy,
                **case_metrics(
                    test_labels / f"{nnunet_case_id}.nii.gz",
                    prediction_folder / f"{nnunet_case_id}.nii.gz",
                ),
            })

    holdout_metrics = pd.DataFrame(metric_records)
    holdout_metrics.to_csv(
        EVALUATION_ROOT / "holdout_case_metrics.csv", index=False
    )
    summaries = {
        f"{split}_{policy}": summarize_metrics(part)
        for (split, policy), part in holdout_metrics.groupby(
            ["split", "inference_policy"]
        )
    }
    (EVALUATION_ROOT / "holdout_summary.json").write_text(
        json.dumps(summaries, indent=2, sort_keys=True) + "\n"
    )
    display(summaries)
else:
    print("Evaluation waits for completed holdout training.")

## 12 — Package the evaluated validation-selected model

The release contains one fold-0 `checkpoint_best.pth` plus the fixed split and
validation/test metrics. Predictions remain drafts requiring native-grid
review.

In [ ]:
if RUN_HOLDOUT_250 and (
    EVALUATION_ROOT / "holdout_summary.json"
).is_file():
    release_parent = WORK / "holdout_inference_release"
    release_name = (
        "nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres"
    )
    release_root = release_parent / release_name
    if release_parent.exists():
        shutil.rmtree(release_parent)
    release_root.mkdir(parents=True)
    source_root = model_root(CV_TRAINER)
    shutil.copy2(source_root / "plans.json", release_root / "plans.json")
    shutil.copy2(
        source_root / "dataset.json", release_root / "dataset.json"
    )
    release_checkpoint_record = portable_checkpoint(
        source_root / "fold_0/checkpoint_best.pth",
        release_root / "fold_0/checkpoint_best.pth",
        "nnUNetTrainer_250epochs",
    )
    release_manifest = {
        **release_checkpoint_record,
        "fold": 0,
        "checkpoint": "checkpoint_best.pth",
        "model_count": 1,
        "training_cases": 16,
        "validation_cases": 2,
        "sealed_test_cases": 2,
        "test_independent_animals": 1,
        "disable_tta": DEPLOY_DISABLE_TTA,
        "postprocessing": "none",
        "predictions_are_drafts": True,
        "parameter_count": plan_summary["parameter_count"],
        "training_manifest_sha256": sha256(TRAINING_MANIFEST),
        "holdout_summary_sha256": sha256(
            EVALUATION_ROOT / "holdout_summary.json"
        ),
    }
    (release_root / "release_manifest.json").write_text(
        json.dumps(release_manifest, indent=2, sort_keys=True) + "\n"
    )
    for path in (
        PROVENANCE / "protocol_identity.json",
        PROVENANCE / "plan_summary.json",
        PROVENANCE / "split_assignments.csv",
        EVALUATION_ROOT / "holdout_summary.json",
        EVALUATION_ROOT / "holdout_case_metrics.csv",
    ):
        destination = release_root / "provenance" / path.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)

    release_zip = (
        WORK / "LYS_T1_brainmask_standard3d_holdout_inference.zip"
    )
    with zipfile.ZipFile(
        release_zip,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as archive:
        for path in sorted(release_parent.rglob("*")):
            if path.is_file() and not path.is_symlink():
                archive.write(path, path.relative_to(release_parent))
    print(
        "Holdout inference release:",
        release_zip,
        f"{release_zip.stat().st_size / 1024**2:.1f} MiB",
    )
    display(FileLink(str(release_zip)))

## 13 — Build review and resume artifacts

The review ZIP excludes MRI arrays. The resume archive retains training
checkpoints but excludes uploaded source images and masks.

In [ ]:
review_zip = WORK / "LYS_T1_brainmask_holdout_16_2_2_review.zip"
allowed_suffixes = {
    ".json", ".csv", ".txt", ".md", ".png", ".pdf", ".py"
}
with zipfile.ZipFile(
    review_zip, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for root in (EXPERIMENT_ROOT, RAW_DATASET, PREPROCESSED_DATASET):
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            if (
                path.is_file()
                and not path.is_symlink()
                and path.suffix.lower() in allowed_suffixes
                and path.stat().st_size <= 25 * 1024 * 1024
            ):
                archive.write(path, path.relative_to(WORK))
print(
    "Review:",
    review_zip,
    f"{review_zip.stat().st_size / 1024**2:.1f} MiB",
)
display(FileLink(str(review_zip)))

if BUILD_RESUME_ARCHIVE:
    resume_tar = (
        WORK / "LYS_T1_brainmask_holdout_16_2_2_resume.tar.gz"
    )
    with tarfile.open(resume_tar, "w:gz", compresslevel=1) as archive:
        if EXPERIMENT_ROOT.exists():
            archive.add(
                EXPERIMENT_ROOT,
                arcname=EXPERIMENT_ROOT.relative_to(WORK),
            )
        results = NNUNET_RESULTS / DATASET_NAME
        if results.exists():
            archive.add(results, arcname=results.relative_to(WORK))
    print(
        "Resume:",
        resume_tar,
        f"{resume_tar.stat().st_size / 1024**3:.2f} GiB",
    )
    display(FileLink(str(resume_tar)))

## Interpretation boundary

- Splits are animal-grouped: 16 scans train, 2 validate, and 2 test.
- The validation scans select `checkpoint_best.pth`.
- Test data are excluded from planning, preprocessing, training, and model
  selection and are evaluated only after training completes.
- Both test scans belong to one animal, so test uncertainty remains high.
- Every prediction is a draft requiring native-grid human review.
- Static pre/post T1 scans do not estimate gadolinium concentration, absolute
  T1, Ktrans, Ki, DCE, or a direct permeability value.